## Test PDF Download + Text Extraction on One Real Document

In [1]:
import requests
from pypdf import PdfReader
from io import BytesIO

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3"
}

url = "https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content"
response = requests.get(url, headers=headers, timeout=20)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Actual bytes downloaded:", len(response.content))

Status code: 200
Content-Type: application/pdf;charset=UTF-8
Actual bytes downloaded: 843539


## Extract Text

In [7]:
pdf_bytes = BytesIO(response.content)
reader = PdfReader(pdf_bytes)

print("Number of pages:", len(reader.pages))
print("---")
print(reader.pages[5].extract_text()[:1000])

Number of pages: 61
---
GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTSFigures
Fig. 1  Analytic framework for antihypertensive medication tr eatment  3
Fig. 2  Framework for analysis  9
Fig. 3  An appr oach for starting treatment with a single-pill combination  26
Fig. 4  An appr oach for starting treatment not using a single-pill combination  
(i.e. with monotherapy or free combination therapy)  27
Fig. 5  Algorithm 1  28
Fig. 6  Algorithm 2  29
Fig. A3.1  Rating of outcomes  43
iv


## Resolve an Old Handle to Its New Bitstream URL via DSpace's REST API

In [8]:
handle = "10665/353829"  # the tuberculosis guideline's old handle

pid_url = f"https://iris.who.int/server/api/pid/find?id=hdl:{handle}"
response = requests.get(pid_url, headers=headers, timeout=20)

print("Status code:", response.status_code)
print(response.text[:1000])

Status code: 200
{
  "id" : "fd4f105d-7b41-488c-985e-0494067c77ef",
  "uuid" : "fd4f105d-7b41-488c-985e-0494067c77ef",
  "name" : "WHO consolidated guidelines on tuberculosis: module 4: treatment: drug-susceptible tuberculosis treatment",
  "handle" : "10665/353829",
  "metadata" : {
    "dc.contributor.author" : [ {
      "value" : "World Health Organization",
      "language" : "en_US",
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.coverage.spatial" : [ {
      "value" : "Geneva",
      "language" : "en_US",
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.accessioned" : [ {
      "value" : "2022-05-04T14:31:25Z",
      "language" : null,
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.available" : [ {
      "value" : "2022-05-25T12:00:00Z",
      "language" : null,
      "authority" : null,
      "confidence" : -1,
      "place" : 0
    } ],
    "dc.date.issue

## Get the Item's Bitstreams

In [9]:
item_uuid = "fd4f105d-7b41-488c-985e-0494067c77ef"

bundles_url = f"https://iris.who.int/server/api/core/items/{item_uuid}/bundles"
response = requests.get(bundles_url, headers=headers, timeout=20)

print("Status code:", response.status_code)
import json
data = response.json()
for bundle in data["_embedded"]["bundles"]:
    print(bundle["name"], "->", bundle["_links"]["bitstreams"]["href"])

Status code: 200
ORIGINAL -> https://iris.who.int/server/api/core/bundles/d3a065ee-38c5-4cff-b5e2-750cff5a1f4d/bitstreams
LICENSE -> https://iris.who.int/server/api/core/bundles/95cad66f-351d-439d-b44d-e15726dcbc99/bitstreams
TEXT -> https://iris.who.int/server/api/core/bundles/b304c3fb-63d8-4dc4-886d-0aa2414acf2e/bitstreams
THUMBNAIL -> https://iris.who.int/server/api/core/bundles/b0d4b621-ac1f-4e6e-80aa-806ba6b75529/bitstreams


## Get the ORIGINAL Bundle's Bitstream (the Real PDF)

In [10]:
original_bitstreams_url = "https://iris.who.int/server/api/core/bundles/d3a065ee-38c5-4cff-b5e2-750cff5a1f4d/bitstreams"
response = requests.get(original_bitstreams_url, headers=headers, timeout=20)

data = response.json()
for bitstream in data["_embedded"]["bitstreams"]:
    print(bitstream["name"])
    print(bitstream["_links"]["content"]["href"])
    print("---")

9789240048126-eng.pdf
https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content
---


## Wrap It Into One Reusable Function

In [11]:
def resolve_who_pdf_url(handle: str, headers: dict) -> str:
    """
    Given an old WHO IRIS handle (e.g., '10665/353829'), resolve it through
    DSpace's REST API to find the actual downloadable PDF URL.
    Returns None if resolution fails at any step.
    """
    # Step 1: handle -> item
    pid_url = f"https://iris.who.int/server/api/pid/find?id=hdl:{handle}"
    r = requests.get(pid_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    item_uuid = r.json()["uuid"]

    # Step 2: item -> bundles
    bundles_url = f"https://iris.who.int/server/api/core/items/{item_uuid}/bundles"
    r = requests.get(bundles_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    bundles = r.json()["_embedded"]["bundles"]
    original_bundle = next((b for b in bundles if b["name"] == "ORIGINAL"), None)
    if not original_bundle:
        return None

    # Step 3: bundle -> bitstreams
    bitstreams_url = original_bundle["_links"]["bitstreams"]["href"]
    r = requests.get(bitstreams_url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None
    bitstreams = r.json()["_embedded"]["bitstreams"]
    if not bitstreams:
        return None

    # Take the first bitstream (usually the main PDF)
    return bitstreams[0]["_links"]["content"]["href"]


# Test it on a handle we haven't tried yet - malaria
test_url = resolve_who_pdf_url("10665/373339", headers=headers)
print(test_url)

https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content


## Extract Handles From Our Curated URLs and Resolve Them All

In [12]:
# topic -> handle (extracted from the URLs we found during research)
who_handles = {
    "tuberculosis": "10665/353829",
    "hypertension": "10665/344424",
    "diabetes": None,  # cdn.who.int URL, not an IRIS handle - handle separately
    "obesity": None,   # cdn.who.int URL, not an IRIS handle - handle separately
    "asthma_copd_pen": "10665/334186",
    "cvd_risk": "10665/43685",
    "pneumonia": "10665/137319",
    "covid19": "10665/365580",
    "malaria": "10665/373339",
    "hiv": None,  # already have direct bitstream URL from search
    "hepatitis_b": "10665/154590",
    "hepatitis_c": "10665/366869",
    "dengue": "10665/76887",
    "typhoid": None,  # already have direct bitstreams URL from search
    "mhgap": "10665/204132",
    "malnutrition": "10665/95584",
    "anemia_pregnancy": "10665/376196",
    "breast_cancer": "10665/137339",
}

resolved_urls = {}
for topic, handle in who_handles.items():
    if handle is None:
        print(f"{topic}: SKIPPED (no handle, needs manual URL)")
        continue
    url = resolve_who_pdf_url(handle, headers=headers)
    resolved_urls[topic] = url
    print(f"{topic}: {url}")

tuberculosis: https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content
hypertension: https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content
diabetes: SKIPPED (no handle, needs manual URL)
obesity: SKIPPED (no handle, needs manual URL)
asthma_copd_pen: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
cvd_risk: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
pneumonia: https://iris.who.int/server/api/core/bitstreams/38cf9b2a-5d7d-49de-a27b-46d5ac4fc387/content
covid19: https://iris.who.int/server/api/core/bitstreams/54fd5754-5ae5-4af1-bc1b-f15960301f17/content
malaria: https://iris.who.int/server/api/core/bitstreams/8fa903e7-5502-4c33-a300-7a3f3469bfab/content
hiv: SKIPPED (no handle, needs manual URL)
hepatitis_b: https://iris.who.int/server/api/core/bitstreams/51bfba1f-fbbe-4ae3-a950-48cf39601916/content
hepatitis_c: https://

## Complete the URL Dictionary + Map to All 36 Project Topics

In [13]:
# Manual URLs for the 4 that weren't IRIS handles
resolved_urls["diabetes"] = "https://cdn.who.int/media/docs/default-source/ncds/ncd-surveillance/guidance-on-global-monitoring-for-diabetes.pdf"
resolved_urls["obesity"] = "https://cdn.who.int/media/docs/default-source/obesity/who-discussion-paper-on-obesity---final190821.pdf"
resolved_urls["hiv"] = "https://iris.who.int/server/api/core/bitstreams/15bfbf7f-9dc6-44fe-8dc5-1beb7be8848b/content"
resolved_urls["typhoid"] = "https://iris.who.int/bitstreams/ebae84fc-9d27-420a-9e6f-41cb85873832/download"

# Map shared documents to every actual project topic (36 total) that they cover.
# Topics not listed here have no dedicated WHO guideline (confirmed gaps).
topic_to_who_doc = {
    "tuberculosis": resolved_urls["tuberculosis"],
    "hypertension": resolved_urls["hypertension"],
    "diabetes": resolved_urls["diabetes"],
    "obesity": resolved_urls["obesity"],
    "asthma": resolved_urls["asthma_copd_pen"],
    "copd": resolved_urls["asthma_copd_pen"],
    "coronary artery disease": resolved_urls["cvd_risk"],
    "heart failure": resolved_urls["cvd_risk"],
    "stroke": resolved_urls["cvd_risk"],
    "hyperlipidemia": resolved_urls["cvd_risk"],
    "pneumonia": resolved_urls["pneumonia"],
    "covid-19": resolved_urls["covid19"],
    "malaria": resolved_urls["malaria"],
    "hiv aids": resolved_urls["hiv"],
    "hepatitis b": resolved_urls["hepatitis_b"],
    "hepatitis c": resolved_urls["hepatitis_c"],
    "dengue fever": resolved_urls["dengue"],
    "typhoid": resolved_urls["typhoid"],
    "depression": resolved_urls["mhgap"],
    "anxiety disorder": resolved_urls["mhgap"],
    "epilepsy": resolved_urls["mhgap"],
    "malnutrition": resolved_urls["malnutrition"],
    "anemia in pregnancy": resolved_urls["anemia_pregnancy"],
    "breast cancer": resolved_urls["breast_cancer"],
}

print(f"Topics with WHO guidance: {len(topic_to_who_doc)}")
print(f"Topics without (confirmed gaps): {36 - len(topic_to_who_doc)}")
for topic, url in topic_to_who_doc.items():
    print(f"  {topic}: {url}")

Topics with WHO guidance: 24
Topics without (confirmed gaps): 12
  tuberculosis: https://iris.who.int/server/api/core/bitstreams/cf34aa08-c5d4-4b64-85cd-2ec1e6b14d91/content
  hypertension: https://iris.who.int/server/api/core/bitstreams/f062769d-f075-4a00-87af-0a2106e0bd04/content
  diabetes: https://cdn.who.int/media/docs/default-source/ncds/ncd-surveillance/guidance-on-global-monitoring-for-diabetes.pdf
  obesity: https://cdn.who.int/media/docs/default-source/obesity/who-discussion-paper-on-obesity---final190821.pdf
  asthma: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
  copd: https://iris.who.int/server/api/core/bitstreams/b9f09202-a320-4c07-ba2c-afe0d1186339/content
  coronary artery disease: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
  heart failure: https://iris.who.int/server/api/core/bitstreams/f106282d-01ad-4978-9a61-27fb1a8c305d/content
  stroke: https://iris.who.int/server/api

## Define the Guideline Data Model

In [15]:
from pydantic import BaseModel
class Guideline(BaseModel):
    title: str
    topic: str
    full_text: str
    num_pages: int
    source_url: str
    source: str = "who"

## Download + Extract Function

In [16]:
def fetch_who_guideline(topic: str, url: str, title: str, headers: dict) -> Guideline:
    """
    Download a WHO guideline PDF and extract its full text.
    """
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    pdf_bytes = BytesIO(response.content)
    reader = PdfReader(pdf_bytes)

    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)

    return Guideline(
        title=title,
        topic=topic,
        full_text=full_text,
        num_pages=len(reader.pages),
        source_url=url,
    )


# Test on tuberculosis
tb_guideline = fetch_who_guideline(
    topic="tuberculosis",
    url=topic_to_who_doc["tuberculosis"],
    title="WHO consolidated guidelines on tuberculosis: Module 4 Treatment",
    headers=headers,
)

print(tb_guideline.title)
print(tb_guideline.num_pages, "pages")
print(len(tb_guideline.full_text), "characters extracted")
print(tb_guideline.full_text[:500])

WHO consolidated guidelines on tuberculosis: Module 4 Treatment
72 pages
145782 characters extracted
WHO 
consolidated 
guidelines on
tuberculosis
Module 4: Treatment
Drug-susceptible 
tuberculosis treatment
WHO consolidated guidelines on tuberculosis   Module 4: Treatment   Drug-susceptible tuberculosis treatment

WHO 
consolidated 
guidelines on
tuberculosis
Module 4: Treatment
Drug-susceptible 
tuberculosis treatment
WHO consolidated guidelines on tuberculosis. Module 4: treatment - drug-susceptible tuberculosis treatment. 
ISBN 978-92-4-004812-6 (electronic  version)
ISBN 978-92-4-004813-3 


## Save to Disk (Same JSONL Pattern)

In [18]:
from pathlib import Path
def save_guideline(guideline: Guideline, output_dir: str = "../data/raw/who"):
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    # one file per topic, since each topic gets exactly one guideline document
    filepath = Path(output_dir) / f"{guideline.topic.replace(' ', '_')}.json"
    
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(guideline.model_dump_json())
    
    return filepath


saved_path = save_guideline(tb_guideline)
print("Saved to:", saved_path.resolve())

Saved to: C:\Users\DELL\Desktop\medrag\data\raw\who\tuberculosis.json


## Full Batch — All 24 Mapped Topics

In [19]:
# Titles for each source document, for clean metadata
who_titles = {
    "tuberculosis": "WHO consolidated guidelines on tuberculosis: Module 4 Treatment",
    "hypertension": "Guideline for the pharmacological treatment of hypertension in adults",
    "diabetes": "Guidance on global monitoring for diabetes prevention and control",
    "obesity": "WHO discussion paper on obesity",
    "asthma": "WHO package of essential noncommunicable (PEN) disease interventions",
    "copd": "WHO package of essential noncommunicable (PEN) disease interventions",
    "coronary artery disease": "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk",
    "heart failure": "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk",
    "stroke": "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk",
    "hyperlipidemia": "Prevention of cardiovascular disease: guidelines for assessment and management of total cardiovascular risk",
    "pneumonia": "Revised WHO classification and treatment of childhood pneumonia at health facilities",
    "covid-19": "Clinical management of COVID-19: living guideline",
    "malaria": "WHO guidelines for malaria",
    "hiv aids": "WHO updated recommendations on HIV clinical management",
    "hepatitis b": "Guidelines for the prevention, care and treatment of persons with chronic hepatitis B infection",
    "hepatitis c": "Guidelines for the care and treatment of persons diagnosed with chronic hepatitis C virus infection",
    "dengue fever": "Handbook for clinical management of dengue",
    "typhoid": "Background document: the diagnosis, treatment and prevention of typhoid fever",
    "depression": "mhGAP guideline for mental, neurological and substance use disorders",
    "anxiety disorder": "mhGAP guideline for mental, neurological and substance use disorders",
    "epilepsy": "mhGAP guideline for mental, neurological and substance use disorders",
    "malnutrition": "Guideline: updates on the management of severe acute malnutrition in infants and children",
    "anemia in pregnancy": "WHO guideline on anaemia",
    "breast cancer": "WHO position paper on mammography screening",
}

import time

results = []
for topic, url in topic_to_who_doc.items():
    try:
        guideline = fetch_who_guideline(topic=topic, url=url, title=who_titles[topic], headers=headers)
        saved_path = save_guideline(guideline)
        print(f"{topic}: saved ({guideline.num_pages} pages, {len(guideline.full_text)} chars)")
        results.append({"topic": topic, "status": "success", "pages": guideline.num_pages})
    except Exception as e:
        print(f"{topic}: FAILED - {e}")
        results.append({"topic": topic, "status": "failed", "error": str(e)})
    time.sleep(1)  # be polite to WHO's servers between downloads

success_count = sum(1 for r in results if r["status"] == "success")
print(f"\n=== DONE === {success_count}/{len(results)} succeeded")

tuberculosis: saved (72 pages, 145782 chars)
hypertension: saved (61 pages, 158048 chars)
diabetes: saved (94 pages, 215398 chars)
obesity: saved (16 pages, 44298 chars)
asthma: saved (85 pages, 223518 chars)
copd: saved (85 pages, 223518 chars)
coronary artery disease: saved (92 pages, 270718 chars)
heart failure: saved (92 pages, 270718 chars)
stroke: saved (92 pages, 270718 chars)
hyperlipidemia: saved (92 pages, 270718 chars)
pneumonia: saved (34 pages, 81499 chars)
covid-19: saved (182 pages, 628183 chars)
malaria: saved (451 pages, 1721974 chars)
hiv aids: saved (108 pages, 356350 chars)
hepatitis b: saved (166 pages, 467646 chars)
hepatitis c: saved (14 pages, 26632 chars)
dengue fever: saved (124 pages, 297755 chars)


invalid pdf header: b'<!DOC'
EOF marker not found


typhoid: FAILED - Stream has ended unexpectedly
depression: saved (71 pages, 165011 chars)
anxiety disorder: saved (71 pages, 165011 chars)
epilepsy: saved (71 pages, 165011 chars)
malnutrition: saved (123 pages, 332057 chars)
anemia in pregnancy: saved (79 pages, 235490 chars)
breast cancer: saved (82 pages, 140790 chars)

=== DONE === 23/24 succeeded


## Retry Typhoid With the Correct URL Format

In [20]:
# Fix: same UUID, correct URL pattern
topic_to_who_doc["typhoid"] = "https://iris.who.int/server/api/core/bitstreams/ebae84fc-9d27-420a-9e6f-41cb85873832/content"

guideline = fetch_who_guideline(
    topic="typhoid",
    url=topic_to_who_doc["typhoid"],
    title=who_titles["typhoid"],
    headers=headers,
)
saved_path = save_guideline(guideline)
print(f"typhoid: saved ({guideline.num_pages} pages, {len(guideline.full_text)} chars)")

typhoid: saved (48 pages, 112073 chars)
